# Notebook 3: Document Chunking

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the third notebook of the project.

In Notebook 1, we collected and standardized the BBC News dataset.

In Notebook 2, we cleaned and preprocessed the article text.

In this notebook, we will split long articles into smaller text chunks.

Chunking is an important step in a RAG pipeline because embedding very long documents directly is not ideal. Smaller chunks help the retriever find more specific and relevant information.

The goal of this notebook is to:

1. Load the preprocessed dataset from Notebook 2.
2. Split each article into smaller overlapping chunks.
3. Store chunk-level metadata.
4. Analyze the chunked dataset.
5. Save the chunked dataset for embedding generation.

The output file from this notebook will be:

`bbc_docs_chunks.csv`

This file will be used in:

`04_Embedding_Generation.ipynb`

In [1]:
# Import required libraries
import os
import pandas as pd



# Find input file automatically
def find_input_file(file_name):
    """
    Searches for the input file in common Kaggle locations.

    This is useful because Kaggle notebooks do not automatically share
    files across different notebooks.
    """
    possible_paths = [
        file_name,
        f"/kaggle/working/{file_name}"
    ]

    # Search inside /kaggle/input
    for root, dirs, files in os.walk("/kaggle/input"):
        if file_name in files:
            possible_paths.append(os.path.join(root, file_name))

    for path in possible_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"{file_name} not found. Please upload it as input to this notebook."
    )



# Load dataset
def load_dataset(file_name):
    """
    Loads the preprocessed dataset from Notebook 2.
    """
    file_path = find_input_file(file_name)
    df = pd.read_csv(file_path)

    print("Dataset loaded successfully.")
    print("Input file:", file_path)
    print("Dataset shape:", df.shape)

    return df


# Split text into chunks
def create_word_chunks(text, chunk_size=180, overlap=40):
    """
    Splits text into overlapping word chunks.

    Parameters:
        text (str): Input article text
        chunk_size (int): Number of words in each chunk
        overlap (int): Number of overlapping words between chunks

    Returns:
        list: List of text chunks
    """
    words = str(text).split()

    if len(words) == 0:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)

        chunks.append(chunk_text)

        # Stop if we reached the end of the article
        if end >= len(words):
            break

        # Move start forward while keeping overlap
        start = end - overlap

    return chunks


# Create chunked dataframe
def build_chunk_dataset(df, text_column="text_clean", chunk_size=180, overlap=40):
    """
    Creates a chunk-level dataset from the document-level dataset.

    Each row in the output dataframe represents one chunk.
    """
    chunk_rows = []

    for _, row in df.iterrows():
        chunks = create_word_chunks(
            text=row[text_column],
            chunk_size=chunk_size,
            overlap=overlap
        )

        for chunk_index, chunk_text in enumerate(chunks):
            chunk_rows.append({
                "chunk_id": f"{row['doc_id']}_{chunk_index + 1}",
                "doc_id": row["doc_id"],
                "chunk_index": chunk_index + 1,
                "title": row["title"],
                "category": row["category"],
                "chunk_text": chunk_text,
                "chunk_word_count": len(chunk_text.split())
            })

    chunk_df = pd.DataFrame(chunk_rows)

    return chunk_df


# Show chunk example
def show_chunk_example(chunk_df):
    """
    Displays one example document and its first few chunks.
    """
    sample_doc_id = chunk_df["doc_id"].sample(1).iloc[0]
    sample_chunks = chunk_df[chunk_df["doc_id"] == sample_doc_id].head(3)

    print("Sample Document ID:", sample_doc_id)
    print("Title:", sample_chunks.iloc[0]["title"])
    print("Category:", sample_chunks.iloc[0]["category"])
    print("\nFirst few chunks from this document:\n")

    for _, row in sample_chunks.iterrows():
        print("=" * 80)
        print("Chunk ID:", row["chunk_id"])
        print("Chunk Index:", row["chunk_index"])
        print("Chunk Word Count:", row["chunk_word_count"])
        print("\nChunk Text Preview:\n")
        print(row["chunk_text"][:700])
        print()


# Save dataset
def save_dataset(df, output_file):
    """
    Saves the chunked dataset as CSV.
    """
    df.to_csv(output_file, index=False)

    print("Dataset saved successfully.")
    print("Output file:", output_file)




In [2]:
# Load preprocessed dataset from Notebook 2
input_file = "bbc_docs_preprocessed.csv"

df_docs = load_dataset(input_file)


# Preview input dataset
print("\nInput Dataset Preview:")
display(df_docs.head())

print("\nInput Dataset Columns:")
print(df_docs.columns.tolist())


# Validate required columns
required_columns = ["doc_id", "title", "category", "text_clean"]

missing_columns = [col for col in required_columns if col not in df_docs.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nAll required columns are available.")


# Create text chunks
CHUNK_SIZE = 180
OVERLAP = 40

df_chunks = build_chunk_dataset(
    df=df_docs,
    text_column="text_clean",
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP
)

print("\nChunking completed successfully.")
print("Chunk size:", CHUNK_SIZE)
print("Overlap:", OVERLAP)
print("Chunked dataset shape:", df_chunks.shape)


# Preview chunked dataset
print("\nChunked Dataset Preview:")
display(df_chunks.head())


# Chunk statistics
print("\nChunk Statistics:")
print("Total documents:", df_chunks["doc_id"].nunique())
print("Total chunks:", len(df_chunks))
print("Average chunks per document:", round(len(df_chunks) / df_chunks["doc_id"].nunique(), 2))

print("\nChunk Word Count Summary:")
print(df_chunks["chunk_word_count"].describe())


# Category-wise chunk distribution
print("\nChunks per Category:")
print(df_chunks["category"].value_counts())


# Show one chunking example
print("\nSample Chunk Example:")
show_chunk_example(df_chunks)


# Save chunked dataset
output_file = "bbc_docs_chunks.csv"

save_dataset(df_chunks, output_file)


# Verify saved file
saved_df = pd.read_csv(output_file)

print("\nSaved file verified successfully.")
print("Saved file shape:", saved_df.shape)

display(saved_df.head())

Dataset loaded successfully.
Input file: /kaggle/input/datasets/jahnavidulala/bbc-docs-preprocessed/bbc_docs_preprocessed.csv
Dataset shape: (8622, 7)

Input Dataset Preview:


,doc_id,title,category,text,text_clean,word_count,clean_word_count
0,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,More than 1.5 million Ukrainians have fled the...,21,21
1,2,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,Russian gymnast Ivan Kuliak is being investiga...,31,31
2,3,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,Several US presidents have failed to get the m...,21,21
3,4,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,A ceasefire agreement in the southern city of ...,20,20
4,5,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,The moment a man swims out of the path of a co...,20,20



Input Dataset Columns:
['doc_id', 'title', 'category', 'text', 'text_clean', 'word_count', 'clean_word_count']

All required columns are available.

Chunking completed successfully.
Chunk size: 180
Overlap: 40
Chunked dataset shape: (8622, 7)

Chunked Dataset Preview:


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20



Chunk Statistics:
Total documents: 8622
Total chunks: 8622
Average chunks per document: 1.0

Chunk Word Count Summary:
count    8622.000000
mean       23.374971
std         3.356254
min        20.000000
25%        21.000000
50%        22.000000
75%        25.000000
max        45.000000
Name: chunk_word_count, dtype: float64

Chunks per Category:
category
unknown    8622
Name: count, dtype: int64

Sample Chunk Example:
Sample Document ID: 1373
Title: Eddie Jones: England head coach admonished by RFU over private school system criticism
Category: unknown

First few chunks from this document:

Chunk ID: 1373_1
Chunk Index: 1
Chunk Word Count: 20

Chunk Text Preview:

England head coach Eddie Jones is admonished by the Rugby Football Union for criticising the system's reliance on private schools.

Dataset saved successfully.
Output file: bbc_docs_chunks.csv

Saved file verified successfully.
Saved file shape: (8622, 7)


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20
